# Automated Search for Short Admissible 3-Successors

Given $A=(a_{-h},\ldots,a_{-1}\mid a_0,\ldots,a_k)$, this notebook enumerates successors
$I(Au\mid Aw)$ with digits in $\{1,2,3\}$ in increasing order of length. It computes the T-ratio of
each interval **directly from the definition** and tests whether the admissible candidates cover $I(A)$.

The search does not use the lists in Lemma 2 or the formula in Hilfssatz 4. Since the computation uses
high-precision numerical arithmetic, the resulting candidates are evidence for a proof, not a rigorous
certificate. Any final proof should recheck them using algebraic numbers or interval arithmetic.

In [15]:
from itertools import product

# Precision used by the numerical search. Increase it to 256 bits if necessary.
RF = RealField(160)
DEFAULT_ADMISSIBLE_BOUNDS = (RF(5) / 17, RF(17) / 5)
DEFAULT_TOL = RF(2) ** (-120)

_cf_cache = {}

def cf_value(blocks):
    """Evaluate a finite/eventually periodic Sage continued fraction over RF."""
    key = tuple(tuple(block) for block in blocks)
    if key not in _cf_cache:
        _cf_cache[key] = RF(continued_fraction(list(key)))
    return _cf_cache[key]

def normalize_word(word, name="word", alphabet=(1, 2, 3, 4)):
    word = tuple(ZZ(a) for a in word)
    bad = [a for a in word if a not in alphabet]
    if bad:
        raise ValueError(f"{name} contains digits outside {alphabet}: {bad}")
    return word

def normalize_A(A):
    if len(A) != 2:
        raise ValueError("A must be a pair (negative_side, positive_side)")
    return (
        normalize_word(A[0], "negative side of A", alphabet=(1, 2, 3, 4)),
        normalize_word(A[1], "positive side of A", alphabet=(1, 2, 3, 4)),
    )

## T-intervals and T-ratios

Following Schecker's definition, the endpoints are computed directly from the two pairings of the periodic
tails $\overline{1,3},\overline{3,1}$ and $\overline{1,2},\overline{2,1}$.

In [16]:
ETA_EXTENSIONS = ((1, 3), (3, 1))
THETA_EXTENSIONS = ((1, 2), (2, 1))

def _cf_candidates(side, extensions):
    return [(cf_value(((0, *side), ext)), ext) for ext in extensions]

def _pick_cf(side, extensions, choose_min):
    candidates = _cf_candidates(side, extensions)
    chooser = min if choose_min else max
    return chooser(candidates, key=lambda pair: pair[0])

def T_interval(negative_side, positive_side):
    """Return (left, right, left_witness, right_witness)."""
    negative_side = tuple(negative_side)
    positive_side = tuple(positive_side)
    left_candidates = []
    right_candidates = []

    for positive_exts, negative_exts in (
        (ETA_EXTENSIONS, THETA_EXTENSIONS),
        (THETA_EXTENSIONS, ETA_EXTENSIONS),
    ):
        p_left, p_left_ext = _pick_cf(positive_side, positive_exts, True)
        n_left, n_left_ext = _pick_cf(negative_side, negative_exts, True)
        left_candidates.append((p_left + n_left, (n_left_ext, p_left_ext)))

        p_right, p_right_ext = _pick_cf(positive_side, positive_exts, False)
        n_right, n_right_ext = _pick_cf(negative_side, negative_exts, False)
        right_candidates.append((p_right + n_right, (n_right_ext, p_right_ext)))

    left, left_witness = min(left_candidates, key=lambda pair: pair[0])
    right, right_witness = max(right_candidates, key=lambda pair: pair[0])
    return left, right, left_witness, right_witness

def T_ratio(negative_side, positive_side):
    """Schecker's T-ratio nu(I(negative_side | positive_side))."""
    p = [cf_value(((0, *positive_side), ext)) for ext in ETA_EXTENSIONS]
    n = [cf_value(((0, *negative_side), ext)) for ext in ETA_EXTENSIONS]
    return abs((p[0] - p[1]) / (n[0] - n[1]))

def admissibility_status(A, admissible_bounds=DEFAULT_ADMISSIBLE_BOUNDS, tol=DEFAULT_TOL):
    """Check all three conditions in Schecker's definition of admissibility."""
    negative_side, positive_side = normalize_A(A)
    ratio = T_ratio(negative_side, positive_side)
    ratio_lo, ratio_hi = admissible_bounds
    negative_shape_ok = len(negative_side) >= 2 or (
        len(negative_side) >= 1 and negative_side[0] >= 2
    )
    positive_shape_ok = len(positive_side) >= 2 or (
        len(positive_side) >= 1 and positive_side[0] >= 2
    )
    ratio_ok = ratio_lo - tol <= ratio <= ratio_hi + tol
    return {
        "admissible": negative_shape_ok and positive_shape_ok and ratio_ok,
        "negative_shape_ok": negative_shape_ok,
        "positive_shape_ok": positive_shape_ok,
        "ratio_ok": ratio_ok,
        "ratio": ratio,
    }

def interval_data(A):
    negative_side, positive_side = normalize_A(A)
    left, right, left_witness, right_witness = T_interval(negative_side, positive_side)
    return {
        "A": (negative_side, positive_side),
        "left": left,
        "right": right,
        "ratio": T_ratio(negative_side, positive_side),
        "left_witness": left_witness,
        "right_witness": right_witness,
    }

## Enumerating Successors and Testing Coverage

The length of a successor is `total_length = len(u) + len(w)`. An empty word is allowed on either side,
but $(u,w)=(\varnothing,\varnothing)$ is excluded because it is not a proper successor. Coverage is tested
with the standard greedy algorithm for one-dimensional interval covering. For a fixed candidate family,
this algorithm returns a cover using the minimum number of intervals.

In [17]:
def words_of_length(length, alphabet=(1, 2, 3)):
    return [tuple(word) for word in product(alphabet, repeat=length)]

def successor_pairs_of_total_length(total_length, alphabet=(1, 2, 3)):
    """All (u,w) with len(u)+len(w)=total_length."""
    if total_length < 1:
        return []
    pairs = []
    for u_length in range(total_length + 1):
        w_length = total_length - u_length
        for u in words_of_length(u_length, alphabet):
            for w in words_of_length(w_length, alphabet):
                pairs.append((u, w))
    return pairs

def make_successor_record(A, successor, admissible_bounds=DEFAULT_ADMISSIBLE_BOUNDS, tol=DEFAULT_TOL):
    negative_side, positive_side = normalize_A(A)
    u = normalize_word(successor[0], "negative successor", alphabet=(1, 2, 3))
    w = normalize_word(successor[1], "positive successor", alphabet=(1, 2, 3))
    if not u and not w:
        raise ValueError("a proper successor must extend at least one side")
    full_A = (negative_side + u, positive_side + w)
    data = interval_data(full_A)
    status = admissibility_status(full_A, admissible_bounds, tol)
    data.update({
        "successor": (u, w),
        "total_length": len(u) + len(w),
        "admissible": status["admissible"],
        "admissibility": status,
    })
    return data

def enumerate_admissible_successors(A, max_total_length, admissible_bounds=DEFAULT_ADMISSIBLE_BOUNDS, tol=DEFAULT_TOL):
    """Enumerate admissible proper 3-successors up to max_total_length."""
    A = normalize_A(A)
    records = []
    for total_length in range(1, max_total_length + 1):
        for successor in successor_pairs_of_total_length(total_length):
            record = make_successor_record(A, successor, admissible_bounds, tol)
            if record["admissible"]:
                records.append(record)
    return records

def clipped_to_target(record, target, tol=DEFAULT_TOL):
    """Clip a successor interval to I(A); return None if it misses I(A)."""
    left = max(record["left"], target["left"])
    right = min(record["right"], target["right"])
    if right < left - tol:
        return None
    clipped = dict(record)
    clipped["cover_left"] = left
    clipped["cover_right"] = right
    return clipped

def greedy_interval_cover(target, records, tol=DEFAULT_TOL):
    """Minimum-cardinality cover of target by records, or an uncovered gap."""
    candidates = [clipped_to_target(record, target, tol) for record in records]
    candidates = [record for record in candidates if record is not None]
    candidates.sort(key=lambda r: (r["cover_left"], -r["cover_right"], r["total_length"]))

    cursor = target["left"]
    chosen = []
    scan = 0
    while cursor < target["right"] - tol:
        best = None
        while scan < len(candidates) and candidates[scan]["cover_left"] <= cursor + tol:
            candidate = candidates[scan]
            if (
                best is None
                or candidate["cover_right"] > best["cover_right"] + tol
                or (
                    abs(candidate["cover_right"] - best["cover_right"]) <= tol
                    and candidate["total_length"] < best["total_length"]
                )
            ):
                best = candidate
            scan += 1

        if best is None or best["cover_right"] <= cursor + tol:
            next_left = candidates[scan]["cover_left"] if scan < len(candidates) else target["right"]
            return {
                "covered": False,
                "chosen": chosen,
                "gap": (cursor, min(next_left, target["right"])),
            }

        chosen.append(best)
        cursor = best["cover_right"]

    return {"covered": True, "chosen": chosen, "gap": None}

def cover_margins(target, chosen):
    """Boundary margins and consecutive overlaps for a chosen cover."""
    ordered = sorted(chosen, key=lambda r: (r["left"], r["right"]))
    if not ordered:
        return None
    return {
        "left_margin": target["left"] - ordered[0]["left"],
        "right_margin": ordered[-1]["right"] - target["right"],
        "overlaps": [ordered[j]["right"] - ordered[j + 1]["left"] for j in range(len(ordered) - 1)],
    }

## Finding the First Depth That Gives a Short Cover

`search_short_admissible_successors(A, max_total_length)` is the main function. It accumulates candidates
at depths $1,2,\ldots$ and stops at the first depth that covers $I(A)$. Set `stop_at_first_cover=False`
to continue through the specified maximum depth and return the final result.

In [18]:
def search_short_admissible_successors(
    A,
    max_total_length=6,
    admissible_bounds=DEFAULT_ADMISSIBLE_BOUNDS,
    tol=DEFAULT_TOL,
    stop_at_first_cover=True,
    verbose=True,
):
    A = normalize_A(A)
    target = interval_data(A)
    target_admissibility = admissibility_status(A, admissible_bounds, tol)
    all_admissible = []
    history = []
    final_cover = {"covered": False, "chosen": [], "gap": (target["left"], target["right"])}
    if verbose and not target_admissibility["admissible"]:
        print("warning: the input A is not admissible; the search will still be performed")

    for depth in range(1, max_total_length + 1):
        new_admissible = []
        for successor in successor_pairs_of_total_length(depth):
            record = make_successor_record(A, successor, admissible_bounds, tol)
            if record["admissible"]:
                new_admissible.append(record)
        all_admissible.extend(new_admissible)
        final_cover = greedy_interval_cover(target, all_admissible, tol)
        step = {
            "depth": depth,
            "new_admissible": len(new_admissible),
            "total_admissible": len(all_admissible),
            "covered": final_cover["covered"],
            "gap": final_cover["gap"],
        }
        history.append(step)
        if verbose:
            if final_cover["covered"]:
                print(
                    f"depth={depth}: admissible {len(all_admissible)}; "
                    f"covered by {len(final_cover['chosen'])} intervals"
                )
            else:
                gap = final_cover["gap"]
                print(
                    f"depth={depth}: admissible {len(all_admissible)}; "
                    f"first uncovered gap = ({gap[0]}, {gap[1]})"
                )
        if final_cover["covered"] and stop_at_first_cover:
            break

    return {
        "A": A,
        "target": target,
        "target_admissibility": target_admissibility,
        "depth": history[-1]["depth"] if history else 0,
        "covered": final_cover["covered"],
        "chosen": final_cover["chosen"],
        "gap": final_cover["gap"],
        "margins": cover_margins(target, final_cover["chosen"]) if final_cover["covered"] else None,
        "admissible_candidates": all_admissible,
        "history": history,
        "admissible_bounds": admissible_bounds,
        "tolerance": tol,
    }

def _word_text(word):
    return "∅" if not word else ",".join(str(a) for a in word)

def show_search_result(result, digits=16):
    target = result["target"]
    print(f"A = ({_word_text(result['A'][0])} | {_word_text(result['A'][1])})")
    print(f"nu(A) = {target['ratio'].n(digits=digits)}")
    print(f"A admissible = {result['target_admissibility']['admissible']}")
    print(f"I(A) = [{target['left'].n(digits=digits)}, {target['right'].n(digits=digits)}]")
    print(f"searched through total length {result['depth']}")

    if not result["covered"]:
        print(f"NOT COVERED; first gap = {result['gap']}")
        return

    print(f"COVERED by {len(result['chosen'])} admissible successors:")
    chosen = sorted(result["chosen"], key=lambda r: (r["left"], r["right"]))
    for record in chosen:
        u, w = record["successor"]
        print(
            f"  I(~{_word_text(u)} | ~{_word_text(w)}), "
            f"length={record['total_length']}, "
            f"nu={record['ratio'].n(digits=digits)}, "
            f"interval=[{record['left'].n(digits=digits)}, {record['right'].n(digits=digits)}]"
        )
    margins = result["margins"]
    print(f"left boundary margin  = {margins['left_margin'].n(digits=digits)}")
    print(f"successive overlaps   = {[x.n(digits=digits) for x in margins['overlaps']]}")
    print(f"right boundary margin = {margins['right_margin'].n(digits=digits)}")

## Example: $A=(3\mid2)$ from the Counterexample

The search should find a cover by admissible successors of length at most two, different from the list in
Lemma 2 of the original paper.

In [19]:
A = ([3], [2])
result = search_short_admissible_successors(A, max_total_length=4)
show_search_result(result)

depth=1: admissible 2; first uncovered gap = (0.72904456331167305882649689027074817702932599647, 0.73882872072542715447506379503575620865220841016)
depth=2: admissible 8; covered by 3 intervals
A = (3 | 2)
nu(A) = 1.958257569495584
A admissible = True
I(A) = [0.6262067619267067, 0.7388287207254272]
searched through total length 2
COVERED by 3 admissible successors:
  I(~∅ | ~1), length=1, nu=0.7654653670707977, interval=[0.6262067619267067, 0.6943902136314009]
  I(~∅ | ~2), length=1, nu=0.3400781479226406, interval=[0.6765338007324686, 0.7290445633116731]
  I(~2 | ~3), length=2, nu=1.130467325807635, interval=[0.7257484011788327, 0.7388287207254272]
left boundary margin  = 0.0000000000000000
successive overlaps   = [0.01785641289893227, 0.003296162132840355]
right boundary margin = 0.0000000000000000


To search for another input, change `A` in the final code cell. For example, the odd-parity counterexample
uses `A = ([3], [1, 1])`. If no cover is found, increase `max_total_length`.

Use `result["admissible_candidates"]` to inspect all admissible candidates and `result["history"]` to
inspect the search status at each depth.

## Surveying Successor-Cover Types for Many Inputs

The following functions run the search for many choices of $A$ and record the input, $h+k$, its parity,
$\nu(A)$, the first successful search depth, and the ordered type of the admissible successors in the cover.
The order in a cover type is the geometric left-to-right order of the corresponding T-intervals.

`enumerate_base_intervals` can generate all base words within specified side-length bounds. Its default
alphabet is $\{1,2,3,4\}$, while all appended successors still use only $\{1,2,3\}$.

In [20]:
def enumerate_base_intervals(
    max_negative_length,
    max_positive_length,
    min_negative_length=1,
    min_positive_length=1,
    alphabet=(1, 2, 3, 4),
    admissible_only=True,
    admissible_bounds=DEFAULT_ADMISSIBLE_BOUNDS,
    tol=DEFAULT_TOL,
):
    """Generate base intervals A in increasing order of h+k."""
    length_pairs = [
        (h, k)
        for h in range(min_negative_length, max_negative_length + 1)
        for k in range(min_positive_length, max_positive_length + 1)
    ]
    length_pairs.sort(key=lambda hk: (hk[0] + hk[1], hk[0], hk[1]))

    for h, k in length_pairs:
        for negative_side in product(alphabet, repeat=h):
            for positive_side in product(alphabet, repeat=k):
                A = (negative_side, positive_side)
                if admissible_only and not admissibility_status(
                    A, admissible_bounds, tol
                )["admissible"]:
                    continue
                yield A

def _ordered_cover_type(search_result):
    ordered = sorted(
        search_result["chosen"],
        key=lambda record: (record["left"], record["right"]),
    )
    return tuple(record["successor"] for record in ordered)

def successor_type_text(cover_type):
    if not cover_type:
        return "NOT COVERED"
    return " ∪ ".join(
        f"I(~{_word_text(u)} | ~{_word_text(w)})" for u, w in cover_type
    )

def survey_successor_cover_types(
    A_values,
    max_total_length=6,
    admissible_A_only=True,
    admissible_bounds=DEFAULT_ADMISSIBLE_BOUNDS,
    tol=DEFAULT_TOL,
    progress_every=None,
    keep_search_results=True,
):
    """Search many inputs and return one record per retained A."""
    records = []
    for input_index, A in enumerate(A_values, start=1):
        A = normalize_A(A)
        input_status = admissibility_status(A, admissible_bounds, tol)
        if admissible_A_only and not input_status["admissible"]:
            continue

        search_result = search_short_admissible_successors(
            A,
            max_total_length=max_total_length,
            admissible_bounds=admissible_bounds,
            tol=tol,
            stop_at_first_cover=True,
            verbose=False,
        )
        h_plus_k = len(A[0]) + len(A[1])
        cover_type = _ordered_cover_type(search_result) if search_result["covered"] else tuple()
        records.append({
            "A": A,
            "A_text": f"({_word_text(A[0])} | {_word_text(A[1])})",
            "h_plus_k": h_plus_k,
            "parity": "even" if h_plus_k % 2 == 0 else "odd",
            "T_ratio": input_status["ratio"],
            "A_admissible": input_status["admissible"],
            "covered": search_result["covered"],
            "search_depth": search_result["depth"],
            "cover_size": len(search_result["chosen"]),
            "cover_type": cover_type,
            "cover_type_text": successor_type_text(cover_type),
            "gap": search_result["gap"],
            "search_result": search_result if keep_search_results else None,
        })

        if progress_every and input_index % progress_every == 0:
            print(f"processed {input_index} inputs; retained {len(records)} records")
    return records

def survey_table(records, digits=12):
    """Return a compact Sage table with one row per input A."""
    rows = [[
        record["A_text"],
        record["h_plus_k"],
        record["parity"],
        str(record["T_ratio"].n(digits=digits)),
        record["search_depth"],
        record["cover_size"],
        record["cover_type_text"],
    ] for record in records]
    return table(rows, header_row=[
        "A", "h+k", "parity", "nu(A)", "depth", "size", "successor-cover type"
    ])

def group_survey_by_cover_type(records):
    """Group records by parity and ordered successor-cover type."""
    groups = {}
    for record in records:
        key = (record["parity"], record["cover_type"])
        groups.setdefault(key, []).append(record)

    summary = []
    for (parity, cover_type), members in groups.items():
        ratios = [record["T_ratio"] for record in members]
        summary.append({
            "parity": parity,
            "cover_type": cover_type,
            "cover_type_text": successor_type_text(cover_type),
            "count": len(members),
            "T_ratio_min": min(ratios),
            "T_ratio_max": max(ratios),
            "examples": [record["A_text"] for record in members],
            "records": members,
        })
    summary.sort(key=lambda group: (
        group["parity"],
        group["T_ratio_min"],
        group["cover_type_text"],
    ))
    return summary

def cover_type_summary_table(summary, digits=12, max_examples=4):
    """Return a Sage table grouped by parity and successor-cover type."""
    rows = [[
        group["parity"],
        group["count"],
        str(group["T_ratio_min"].n(digits=digits)),
        str(group["T_ratio_max"].n(digits=digits)),
        group["cover_type_text"],
        ", ".join(group["examples"][:max_examples]),
    ] for group in summary]
    return table(rows, header_row=[
        "parity", "count", "min nu(A)", "max nu(A)", "successor-cover type", "examples"
    ])

### Example Survey

The explicit list below is inexpensive and convenient for experimentation. To run an exhaustive finite survey,
replace `A_values` by, for example,
`enumerate_base_intervals(max_negative_length=2, max_positive_length=2)`.

In [21]:
A_values = [
    ([3], [2]),
    ([2], [3]),
    ([3], [1, 1]),
    ([1, 1], [3]),
]

survey = survey_successor_cover_types(A_values, max_total_length=4)
survey_table(survey)

A,h+k,parity,nu(A),depth,size,successor-cover type
(3 | 2),\(2\),even,1.95825756950,\(2\),\(3\),I(~∅ | ~1) ∪ I(~∅ | ~2) ∪ I(~2 | ~3)
(2 | 3),\(2\),even,0.510658054169,\(2\),\(3\),I(~1 | ~∅) ∪ I(~2 | ~∅) ∪ I(~3 | ~2)
"(3 | 1,1)",\(3\),odd,1.95825756950,\(2\),\(3\),I(~1 | ~3) ∪ I(~∅ | ~2) ∪ I(~∅ | ~1)
"(1,1 | 3)",\(3\),odd,0.510658054169,\(2\),\(3\),I(~3 | ~1) ∪ I(~2 | ~∅) ∪ I(~1 | ~∅)


In [22]:
cover_type_summary = group_survey_by_cover_type(survey)
cover_type_summary_table(cover_type_summary)

parity,count,min nu(A),max nu(A),successor-cover type,examples
even,\(1\),0.510658054169,0.510658054169,I(~1 | ~∅) ∪ I(~2 | ~∅) ∪ I(~3 | ~2),(2 | 3)
even,\(1\),1.95825756950,1.95825756950,I(~∅ | ~1) ∪ I(~∅ | ~2) ∪ I(~2 | ~3),(3 | 2)
odd,\(1\),0.510658054169,0.510658054169,I(~3 | ~1) ∪ I(~2 | ~∅) ∪ I(~1 | ~∅),"(1,1 | 3)"
odd,\(1\),1.95825756950,1.95825756950,I(~1 | ~3) ∪ I(~∅ | ~2) ∪ I(~∅ | ~1),"(3 | 1,1)"


## Exhaustive Survey of All $A$ up to a Given Depth

Here the depth of a base interval is defined as $h+k$, the total number of fixed digits on its two sides.
`run_exhaustive_A_survey(A_depth, successor_depth)` enumerates every $A$ with
$2\le h+k\le\texttt{A_depth}$, retains the admissible inputs, searches for a cover using successors up to
`successor_depth`, and summarizes all observed cover types.

By default, a cover type and its left-right reflection are identified. This removes the duplication produced
by replacing $A=(a^-\mid a^+)$ with $(a^+\mid a^-)$, which also replaces $\nu(A)$ by $1/\nu(A)$.
Set `identify_reflections=False` if the two orientations should be recorded separately.

In [23]:
def raw_base_interval_count(A_depth, min_depth=2, alphabet=(1, 2, 3, 4)):
    """Number of base intervals before applying admissibility filters."""
    q = len(alphabet)
    return sum((total_depth - 1) * q ** total_depth for total_depth in range(min_depth, A_depth + 1))

def enumerate_base_intervals_up_to_depth(
    A_depth,
    min_depth=2,
    alphabet=(1, 2, 3, 4),
    admissible_only=True,
    admissible_bounds=DEFAULT_ADMISSIBLE_BOUNDS,
    tol=DEFAULT_TOL,
):
    """Generate all A with min_depth <= h+k <= A_depth."""
    if min_depth < 2:
        raise ValueError("min_depth must be at least 2 because both sides are nonempty")
    if A_depth < min_depth:
        return

    for total_depth in range(min_depth, A_depth + 1):
        for h in range(1, total_depth):
            k = total_depth - h
            for negative_side in product(alphabet, repeat=h):
                for positive_side in product(alphabet, repeat=k):
                    A = (negative_side, positive_side)
                    if admissible_only and not admissibility_status(
                        A, admissible_bounds, tol
                    )["admissible"]:
                        continue
                    yield A

def reflect_cover_type(cover_type):
    """Swap the negative and positive successor words."""
    return tuple((w, u) for u, w in cover_type)

def canonical_cover_type(cover_type, identify_reflections=True):
    """Choose a canonical representative, optionally modulo left-right reflection."""
    cover_type = tuple((tuple(u), tuple(w)) for u, w in cover_type)
    if not identify_reflections or not cover_type:
        return cover_type
    reflected = reflect_cover_type(cover_type)
    return min(cover_type, reflected)

def group_exhaustive_survey(records, identify_reflections=True):
    """Group exhaustive-survey records by parity, coverage status, and canonical type."""
    groups = {}
    for record in records:
        canonical_type = canonical_cover_type(
            record["cover_type"], identify_reflections=identify_reflections
        )
        key = (record["parity"], record["covered"], canonical_type)
        groups.setdefault(key, []).append(record)

    summary = []
    for (parity, covered, cover_type), members in groups.items():
        ratios = [record["T_ratio"] for record in members]
        depths = [record["search_depth"] for record in members]
        summary.append({
            "parity": parity,
            "covered": covered,
            "cover_type": cover_type,
            "cover_type_text": successor_type_text(cover_type),
            "count": len(members),
            "T_ratio_min": min(ratios),
            "T_ratio_max": max(ratios),
            "search_depth_min": min(depths),
            "search_depth_max": max(depths),
            "examples": [record["A_text"] for record in members],
            "records": members,
        })
    summary.sort(key=lambda group: (
        group["parity"],
        not group["covered"],
        group["T_ratio_min"],
        group["cover_type_text"],
    ))
    return summary

def run_exhaustive_A_survey(
    A_depth,
    successor_depth=4,
    min_A_depth=2,
    alphabet=(1, 2, 3, 4),
    identify_reflections=True,
    admissible_bounds=DEFAULT_ADMISSIBLE_BOUNDS,
    tol=DEFAULT_TOL,
    progress_every=100,
    keep_search_results=False,
):
    """Exhaustively survey admissible A up to total depth A_depth."""
    raw_count = raw_base_interval_count(A_depth, min_A_depth, alphabet)
    print(
        f"enumerating {raw_count} base intervals with "
        f"{min_A_depth} <= h+k <= {A_depth}"
    )
    A_values = enumerate_base_intervals_up_to_depth(
        A_depth=A_depth,
        min_depth=min_A_depth,
        alphabet=alphabet,
        admissible_only=True,
        admissible_bounds=admissible_bounds,
        tol=tol,
    )
    records = survey_successor_cover_types(
        A_values,
        max_total_length=successor_depth,
        admissible_A_only=False,
        admissible_bounds=admissible_bounds,
        tol=tol,
        progress_every=progress_every,
        keep_search_results=keep_search_results,
    )
    summary = group_exhaustive_survey(
        records, identify_reflections=identify_reflections
    )
    covered_count = sum(record["covered"] for record in records)
    result = {
        "A_depth": A_depth,
        "min_A_depth": min_A_depth,
        "successor_depth": successor_depth,
        "identify_reflections": identify_reflections,
        "raw_A_count": raw_count,
        "admissible_A_count": len(records),
        "covered_count": covered_count,
        "uncovered_count": len(records) - covered_count,
        "type_count": len(summary),
        "records": records,
        "summary": summary,
    }
    print(
        f"admissible A: {result['admissible_A_count']}; "
        f"covered: {result['covered_count']}; "
        f"not covered through successor depth {successor_depth}: {result['uncovered_count']}; "
        f"grouped types: {result['type_count']}"
    )
    return result

def exhaustive_summary_table(exhaustive_result, digits=12, max_examples=5):
    """Return the parity/type summary table for an exhaustive run."""
    rows = [[
        group["parity"],
        "yes" if group["covered"] else "no",
        group["count"],
        str(group["T_ratio_min"].n(digits=digits)),
        str(group["T_ratio_max"].n(digits=digits)),
        f"{group['search_depth_min']}..{group['search_depth_max']}",
        group["cover_type_text"],
        ", ".join(group["examples"][:max_examples]),
    ] for group in exhaustive_result["summary"]]
    return table(rows, header_row=[
        "parity", "covered", "count", "min nu(A)", "max nu(A)",
        "successor depth", "canonical successor-cover type", "examples"
    ])

### Example Exhaustive Run

The example below examines every admissible $A$ with $h+k\le3$. Increase `A_depth` cautiously: before
admissibility filtering, depth $d$ contributes $(d-1)4^d$ inputs. `successor_depth` is a separate cutoff
for the successors used to cover each input.

In [26]:
exhaustive_result = run_exhaustive_A_survey(
    A_depth=5,
    successor_depth=6,
    progress_every=50,
)
exhaustive_summary_table(exhaustive_result)

enumerating 5008 base intervals with 2 <= h+k <= 5
processed 50 inputs; retained 50 records
processed 100 inputs; retained 100 records
processed 150 inputs; retained 150 records
processed 200 inputs; retained 200 records
processed 250 inputs; retained 250 records
processed 300 inputs; retained 300 records
processed 350 inputs; retained 350 records
processed 400 inputs; retained 400 records
processed 450 inputs; retained 450 records
processed 500 inputs; retained 500 records
processed 550 inputs; retained 550 records
processed 600 inputs; retained 600 records
processed 650 inputs; retained 650 records
processed 700 inputs; retained 700 records
processed 750 inputs; retained 750 records
processed 800 inputs; retained 800 records
processed 850 inputs; retained 850 records
admissible A: 855; covered: 855; not covered through successor depth 6: 0; grouped types: 41


parity,covered,count,min nu(A),max nu(A),successor depth,canonical successor-cover type,examples
even,yes,\(12\),0.297457431219,3.36182557586,1..1,I(~∅ | ~3) ∪ I(~∅ | ~2) ∪ I(~∅ | ~1),"(1,1 | 1,3), (1,3 | 1,1), (2,1 | 4,1), (2,2 | 4,2), (2,3 | 3,4)"
even,yes,\(2\),0.302343026141,3.30750145874,2..2,I(~2 | ~3) ∪ I(~1 | ~3) ∪ I(~∅ | ~2) ∪ I(~∅ | ~1),"(1,4 | 2,4), (2,4 | 1,4)"
even,yes,\(2\),0.309307341416,3.23303027798,1..1,I(~∅ | ~1) ∪ I(~∅ | ~2) ∪ I(~∅ | ~3),"(2 | 4), (4 | 2)"
even,yes,\(2\),0.316209867329,3.16245665718,4..4,"I(~∅ | ~1) ∪ I(~∅ | ~2) ∪ I(~1 | ~3) ∪ I(~2,3 | ~3,3) ∪ I(~2 | ~3)","(4 | 1,1,3), (1,1,3 | 4)"
even,yes,\(8\),0.316209867329,3.16245665718,4..4,"I(~2 | ~3) ∪ I(~2,3 | ~3,3) ∪ I(~1 | ~3) ∪ I(~∅ | ~2) ∪ I(~∅ | ~1)","(1,3 | 2,3), (2,3 | 1,3), (3,2 | 3,4), (3,3 | 4,1), (3,4 | 3,2)"
even,yes,\(8\),0.340078147923,2.94050060584,3..3,"I(~2 | ~3) ∪ I(~3 | ~2,1) ∪ I(~∅ | ~2) ∪ I(~∅ | ~1)","(1,2 | 2,2), (2,2 | 1,2), (2,2 | 2,4), (2,4 | 2,2), (3,3 | 4,4)"
even,yes,\(2\),0.340078147923,2.94050060584,3..3,"I(~∅ | ~1) ∪ I(~∅ | ~2) ∪ I(~3 | ~2,1) ∪ I(~2 | ~3)","(3 | 1,1,2), (1,1,2 | 3)"
even,yes,\(46\),0.390891054882,2.55825756950,2..2,I(~2 | ~3) ∪ I(~∅ | ~2) ∪ I(~∅ | ~1),"(1,1 | 1,2), (1,1 | 2,1), (1,2 | 1,1), (1,2 | 1,4), (1,2 | 3,1)"
even,yes,\(12\),0.390891054882,2.55825756950,2..2,I(~∅ | ~1) ∪ I(~∅ | ~2) ∪ I(~2 | ~3),"(2 | 3), (3 | 2), (2 | 1,1,1), (3 | 1,2,1), (4 | 1,1,2)"
even,yes,\(22\),0.582354789439,1.71716626726,2..2,I(~2 | ~3) ∪ I(~3 | ~2) ∪ I(~1 | ~3) ∪ I(~∅ | ~1),"(1,2 | 1,3), (1,3 | 1,2), (1,3 | 1,4), (1,4 | 1,3), (1,4 | 4,1)"
